# Task 2
This serves as a template which will guide you through the implementation of this task. It is advised to first read the whole template and get a sense of the overall structure of the code before trying to fill in any of the TODO gaps.
This is the jupyter notebook version of the template. For the python file version, please refer to the file `template_solution.py`.

First, we import necessary libraries:

In [60]:
import numpy as np
import pandas as pd

# Add any other imports you need here
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import DotProduct, RBF, Matern, RationalQuadratic
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import cross_val_score

# Data Loading
TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test
(and potentially change initialization of variables to accomodate how you deal with non-numeric data)

In [61]:
"""
This loads the training and test data, preprocesses it, removes the NaN
values and interpolates the missing data using imputation

Parameters
----------
Compute
----------
X_train: matrix of floats, training input with features
y_train: array of floats, training output with labels
X_test: matrix of floats: dim = (100, ?), test input with features
"""
# Load training data
train_df = pd.read_csv("../train.csv")
    
print("Training data:")
print("Shape:", train_df.shape)
print(train_df.head(2))
print('\n')
    
# Load test data
test_df = pd.read_csv("../test.csv")

print("Test data:")
print(test_df.shape)
print(test_df.head(2))

# Dummy initialization of the X_train, X_test and y_train   
# TODO: Depending on how you deal with the non-numeric data, you may want to 
# modify/ignore the initialization of these variables

# Drop rows where no value for CHF is given
train_df = train_df[train_df["price_CHF"].notna()]

# Split into y train
y_train = train_df['price_CHF']

# Apply the iterative imputer grouped by seasons
seasons = ["spring", "summer", "autumn", "winter"]

# List to hold the imputed season for train and test data
train_parts = []
test_parts = []

# Iterate over the seasons
for season in seasons:
    # Filter out current season
    train_season = train_df[train_df["season"] == season]
    test_season = test_df[test_df["season"] == season]

    # Drop not needed cols
    train_features = train_season.drop(columns=["season", "price_CHF"])
    test_features = test_season.drop(columns=["season"])

    # Fit an imputer only on the current seaons and apply it to the test data
    imp = IterativeImputer(max_iter=10, random_state=0)
    train_imputed = imp.fit_transform(train_features)
    test_imputed = imp.transform(test_features)

    # Convert back to a df
    train_part = pd.DataFrame(train_imputed, columns=train_features.columns, index=train_season.index)
    test_part = pd.DataFrame(test_imputed, columns=test_features.columns, index=test_season.index)

    # Add season back as col
    train_part["season"] = season
    test_part["season"] = season

    train_parts.append(train_part)
    test_parts.append(test_part)

# Combine the parts back together and get right order as before
train_combined = pd.concat(train_parts).sort_index()
test_combined = pd.concat(test_parts).sort_index()

# Convert the seasons into numerical values
X_train_num = pd.get_dummies(train_combined, columns=["season"], dtype=int)
X_test_num = pd.get_dummies(test_combined, columns=["season"], dtype=int)


# TODO: Perform data preprocessing, imputation and extract X_train, y_train and X_test
X_train = X_train_num.to_numpy()
y_train = y_train.to_numpy()
X_test = X_test_num.to_numpy()

assert (X_train.shape[1] == X_test.shape[1]) and (X_train.shape[0] == y_train.shape[0]) and (X_test.shape[0] == 100), "Invalid data shape"

Training data:
Shape: (900, 11)
   season  price_AUS  price_CHF  price_CZE  price_GER  price_ESP  price_FRA  \
0  spring  -3.348808        NaN  -3.597534  -4.102160  -2.201652  -2.806995   
1  summer  -3.421345  -1.455502  -3.597649  -3.675204        NaN  -2.440406   

   price_UK  price_ITA  price_POL  price_SVK  
0       NaN   -3.61728  -2.758448        NaN  
1 -2.379524        NaN        NaN   -3.72506  


Test data:
(100, 10)
   season  price_AUS  price_CZE  price_GER  price_ESP  price_FRA  price_UK  \
0  spring  -1.504285  -1.632302  -2.347618        NaN        NaN -3.437325   
1  summer  -1.779837  -1.750216  -2.407555  -1.875685        NaN       NaN   

   price_ITA  price_POL  price_SVK  
0  -3.505886  -2.042408        NaN  
1  -3.528359  -2.131659  -2.911154  


In [62]:
"""
Notes for data processing:
- Use iterative imputer as mentioned in the description on train and test data
    - Maybe later to refine the training, apply iterative imputer only on rows from the same season?
- But drop any row where no value for CHF is given, since nothing can be learned from that
"""

# # Convert the season into numerical values
# test_train_df = train_df.copy()
# test_train_df_dumm = pd.get_dummies(test_train_df, columns=["season"], dtype=int)

# # Drop rows where no value for CHF is given
# test_train_df_dumm = test_train_df_dumm[test_train_df_dumm["price_CHF"].notna()]

# # Applying iterative imputer to handle nan in train data
# imp = IterativeImputer(max_iter=10, random_state=0)
# test_train_df_imp = imp.fit_transform(test_train_df_dumm)

# # Convert back into a df
# test_train_df_clean = pd.DataFrame(
#     data=test_train_df_imp, 
#     columns=test_train_df_dumm.columns, 
#     index=test_train_df_dumm.index
# )

# test_train_df_clean


'\nNotes for data processing:\n- Use iterative imputer as mentioned in the description on train and test data\n    - Maybe later to refine the training, apply iterative imputer only on rows from the same season?\n- But drop any row where no value for CHF is given, since nothing can be learned from that\n'

# Modeling and Prediction
TODO: Define the model and fit it using training data. Then, use test data to make predictions

In [63]:
"""
This defines the model, fits training data and then does the prediction
with the test data 

Parameters
----------
X_train: matrix of floats, training input with 10 features
y_train: array of floats, training output
X_test: matrix of floats: dim = (100, ?), test input with 10 features

Compute
----------
y_test: array of floats: dim = (100,), predictions on test set
"""
class Model(object):
    def __init__(self, kernel=DotProduct()):
        super().__init__()
        self._x_train = None
        self._y_train = None

        # Allow to choose which kernel is used and add variance with alpha
        self.gpr = GaussianProcessRegressor(kernel=kernel, alpha=0.02, normalize_y=True)

        # Initate a scaler to better handle the data
        self.scaler = RobustScaler()

    def fit(self, X_train: np.ndarray, y_train: np.ndarray):
        #TODO: Define the model and fit it using (X_train, y_train)
        self._x_train = X_train
        self._y_train = y_train
        # Use the scaler to transfrom and fit the data
        X_train_scaled = self.scaler.fit_transform(X_train)
        self.gpr.fit(X_train_scaled, y_train)

    def predict(self, X_test: np.ndarray) -> np.ndarray:
        #TODO: Use the model to make predictions y_pred using test data X_test
        # Use the scaler to transform the data
        X_test_scaled = self.scaler.transform(X_test)
        y_pred = self.gpr.predict(X_test_scaled)

        assert y_pred.shape == (X_test.shape[0],), "Invalid data shape"
        return y_pred

In [64]:
# Create the scaled version of X_train for testing
cv_scaler = RobustScaler()
X_train_cv_scaled = cv_scaler.fit_transform(X_train)

# Different alpha values to test
for j in [0.01, 0.02, 0.03, 0.04, 0.05]:
    print("Current alpha value:", j)
    
    # Test how well the different kernel fit the train data
    for i in [DotProduct(), RBF(), Matern(), RationalQuadratic()]:
        
        # Add variance with alpha
        test_gpr = GaussianProcessRegressor(kernel=i, alpha=j, normalize_y=True)
        
        # Run cross-validation
        scores = cross_val_score(test_gpr, X_train_cv_scaled, y_train, cv=5, scoring='r2')
        
        print(i)
        print(scores.mean())


Current alpha value: 0.01
DotProduct(sigma_0=1)
0.68208873605421


c:\Users\dario\anaconda3\envs\IML\lib\site-packages\sklearn\gaussian_process\_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 5 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)


RBF(length_scale=1)
0.925324998949739
Matern(length_scale=1, nu=1.5)
0.9278480614426261
RationalQuadratic(alpha=1, length_scale=1)
0.9245077955315709
Current alpha value: 0.02
DotProduct(sigma_0=1)
0.6819287554647232
RBF(length_scale=1)
0.9307100914314382
Matern(length_scale=1, nu=1.5)
0.9296286419574763
RationalQuadratic(alpha=1, length_scale=1)
0.9298566707031892
Current alpha value: 0.03
DotProduct(sigma_0=1)
0.6817526500289267
RBF(length_scale=1)
0.9302795708498749
Matern(length_scale=1, nu=1.5)
0.9287982885904299
RationalQuadratic(alpha=1, length_scale=1)
0.929845507129475
Current alpha value: 0.04
DotProduct(sigma_0=1)
0.6815620078654743
RBF(length_scale=1)
0.9288211432427935
Matern(length_scale=1, nu=1.5)
0.9273384385662498
RationalQuadratic(alpha=1, length_scale=1)
0.9284537046243597
Current alpha value: 0.05
DotProduct(sigma_0=1)
0.6813582705352497
RBF(length_scale=1)
0.9269539681514782
Matern(length_scale=1, nu=1.5)
0.9257796694921184
RationalQuadratic(alpha=1, length_scale=1

In [65]:
model = Model(RBF())
# Use this function to fit the model
model.fit(X_train=X_train, y_train=y_train)
# Use this function for inference
y_pred = model.predict(X_test)

# Saving Results
You don't have to change this

In [66]:
dt = pd.DataFrame(y_pred) 
dt.columns = ['price_CHF']
dt.to_csv('results_df.csv', index=False)
print("\nResults file successfully generated!")


Results file successfully generated!
